# L'or dans huit devises · *Gold in eight currencies*

Notebook compagnon de l'enquête **L'or monte-t-il parce que les monnaies s'effondrent ?** — [lire l'article](https://nmlab.io/ressources/prix-de-l-or-et-effondrement-des-monnaies).
Companion notebook to the study **Is gold rising because currencies are collapsing?**.

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données publiques du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's public data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


import io
import re
import urllib.request
from functools import lru_cache

import numpy as np
import pandas as pd
from pandas import DataFrame, Series

CMO_PAGE = "https://www.worldbank.org/en/research/commodity-markets"
CMO_FILE = ("https://thedocs.worldbank.org/en/doc/74e8be41ceb20fa0da750cda2f6b9e4e-0050012026"
            "/related/CMO-Historical-Data-Monthly.xlsx")
FRED_CSV = "https://fred.stlouisfed.org/graph/fredgraph.csv?id={}"

# H.10 : EUR, GBP et AUD sont cotés en dollars par unité étrangère — on les inverse
# pour obtenir partout des unités locales par dollar, comme dans l'article.
# H.10 quotes EUR, GBP and AUD as dollars per foreign unit: invert them so every
# series is local units per dollar, as in the article.
FX = {"EUR": ("DEXUSEU", True), "JPY": ("DEXJPUS", False), "GBP": ("DEXUSUK", True),
      "CHF": ("DEXSZUS", False), "CAD": ("DEXCAUS", False), "AUD": ("DEXUSAL", True),
      "CNY": ("DEXCHUS", False)}


def _fetch(url: str, tries: int = 4) -> bytes:
    """Télécharge une URL, avec quelques reprises (FRED coupe parfois la connexion).
    Download a URL, retrying a few times (FRED occasionally drops the connection)."""
    import time
    for attempt in range(tries):
        try:
            return urllib.request.urlopen(url, timeout=90).read()
        except Exception:
            if attempt == tries - 1:
                raise
            time.sleep(2 * (attempt + 1))
    raise RuntimeError("unreachable")


@lru_cache(maxsize=None)
def load_gold_usd() -> Series:
    """Or en dollars par once, moyennes mensuelles depuis 1960.

    Source : « Commodity Price Data » (Pink Sheet) de la Banque mondiale, feuille
    « Monthly Prices », colonne Gold — le fixing de Londres, en accès libre.
    World Bank Pink Sheet, monthly London gold price in US dollars per troy ounce.
    """
    try:
        raw = _fetch(CMO_FILE)
    except Exception:                                  # millésime renouvelé : on relit le lien
        page = _fetch(CMO_PAGE).decode("utf-8", "ignore")
        link = re.search(r"https://[^\"']*CMO-Historical-Data-Monthly\.xlsx", page)
        raw = _fetch(link.group(0))
    table = pd.read_excel(io.BytesIO(raw), sheet_name="Monthly Prices", skiprows=4)
    table = table.rename(columns={table.columns[0]: "date"})[["date", "Gold"]].dropna()
    dates = pd.to_datetime(table["date"].str.replace("M", "-"), format="%Y-%m")
    return Series(table["Gold"].values, index=dates).astype(float)


@lru_cache(maxsize=None)
def load_fred(series_id: str) -> Series:
    """Série FRED (CSV public, sans clé) ramenée à des moyennes mensuelles.
    A FRED series (public CSV, no key) averaged to monthly frequency."""
    table = pd.read_csv(io.StringIO(_fetch(FRED_CSV.format(series_id)).decode()))
    values = pd.to_numeric(table[table.columns[1]], errors="coerce")
    series = Series(values.values, index=pd.to_datetime(table[table.columns[0]])).dropna()
    return series.resample("MS").mean()


def load_gold_in_currencies(start: str, end: str) -> DataFrame:
    """Prix de l'or dans les huit devises du panier, mois par mois.

    Chaque prix local est le produit de la moyenne mensuelle de l'or en dollars
    et de la moyenne mensuelle du taux de change — l'ordre des opérations retenu
    par l'article. Le dollar vaut 1 par construction.
    Gold priced in the eight basket currencies, month by month.
    """
    gold = load_gold_usd()
    prices = {"USD": gold}
    for code, (series_id, invert) in FX.items():
        rate = load_fred(series_id)
        prices[code] = gold * (1 / rate if invert else rate)
    return DataFrame(prices).loc[start:end].dropna()


def effective_index(prices: DataFrame) -> Series:
    """Indice or effectif : moyenne géométrique équipondérée des huit prix locaux,
    base 100 au premier mois. Seule la moyenne géométrique garantit que l'indice
    des devises mesurées contre l'or est exactement l'inverse de celui-ci.
    Equal-weighted geometric mean of the eight local prices, first month = 100.
    """
    return 100 * np.exp(np.log(prices / prices.iloc[0]).mean(axis=1))


from matplotlib.figure import Figure
from matplotlib.ticker import FixedLocator, FuncFormatter

START, END = "1999-01-01", "2026-05-01"

LABELS = {
    "fr": dict(
        title="L'or a monté dans les huit devises, sans exception",
        sub="Prix de l'once dans chaque devise, base 100 en janvier 1999 — échelle logarithmique.",
        eff="Indice or effectif", high="JPY — le plus haut", low="CHF — le plus bas",
        others="Les six autres devises du panier",
        note="Le faisceau reste serré : l'écart entre la devise la plus faible et la plus forte est petit devant\n"
             "la hausse commune. Sources : Banque mondiale (or, fixing de Londres) ; Réserve fédérale, H.10 (change)."),
    "en": dict(
        title="Gold rose in all eight currencies, without exception",
        sub="Price of an ounce in each currency, January 1999 = 100 — logarithmic scale.",
        eff="Effective gold index", high="JPY — highest", low="CHF — lowest",
        others="The six other basket currencies",
        note="The bundle stays tight: the gap between the weakest and the strongest currency is small next to the\n"
             "common rise. Sources: World Bank (gold, London fixing); Federal Reserve, H.10 (exchange rates)."),
}


def build_figure(prices: DataFrame, lang: str) -> Figure:
    """Huit courbes en base 100, les deux extrêmes mises en avant, indice effectif en blanc."""
    text = LABELS[lang]
    base = 100 * prices / prices.iloc[0]
    index = effective_index(prices)
    final = base.iloc[-1].sort_values(ascending=False)
    high, low = final.index[0], final.index[-1]

    fig = nm.figure(height_px=1120)
    ax = nm.axes(fig, left=0.062, right=0.982)
    for rank, code in enumerate(c for c in base.columns if c not in (high, low)):
        ax.plot(base.index, base[code], color=nm.COLORS["blue"], lw=2.0, alpha=0.5, zorder=2,
                label=text["others"] if rank == 0 else None)
    ax.plot(base.index, base[high], color=nm.COLORS["rose"], lw=3.0, zorder=4, label=text["high"])
    ax.plot(base.index, base[low], color=nm.COLORS["teal"], lw=3.0, zorder=4, label=text["low"])
    ax.plot(index.index, index, color=nm.COLORS["text"], lw=3.6, zorder=5, label=text["eff"])

    ax.set_yscale("log")
    ax.yaxis.set_major_locator(FixedLocator([100, 200, 500, 1000, 2000]))
    ax.yaxis.set_major_formatter(FuncFormatter(
        lambda v, _: f"{v:,.0f}".replace(",", " " if lang == "fr" else ",")))
    ax.yaxis.set_minor_locator(FixedLocator([]))
    ax.set_ylim(72, 3050)

    # Le faisceau reste anonyme — c'est le propos : les six autres courbes sont
    # indiscernables. Seuls les deux extrêmes et l'indice sont nommés.
    handles = dict(zip(*ax.get_legend_handles_labels()[::-1]))      # libellé → tracé
    order = [text["eff"], text["high"], text["others"], text["low"]]
    legend = ax.legend([handles[label] for label in order], order, loc="upper left",
                       frameon=False, fontsize=20.5, labelcolor="linecolor",
                       handlelength=1.6, borderaxespad=1.2)
    for handle in legend.get_lines():
        handle.set_linewidth(3.4)
        handle.set_alpha(1)

    nm.header(fig, text["title"], text["sub"])
    nm.footer(fig, text["note"])
    return fig


build_figure(load_gold_in_currencies(START, END), LANG)